# Plots maps carbon densities (figure m01 and m03)

In [ ]:
# Libraries
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_nearth = '../data/naturalearth/'
dir_zen = '../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

---

In [ ]:
# Get cspatial carbon density data
ds_in = xr.open_dataset(f'{dir_zen}ecosystem_carbon_cspatial_v1-0.nc')

# Get coastline data
# Download: https://www.naturalearthdata.com/
coastline110 = gpd.read_file(
    f'{dir_nearth}ne_110m_coastline/ne_110m_coastline.shp')

In [ ]:
# Create plots and export
def plot_maps(ctype, cmap):

    # Ecosystem types and order
    list_etype = ['trop_ef', 'trop_df', 'temp_dbf', 'temp_dnf', 'temp_ebf', 
                  'temp_enf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
                  'glob_mount']

    # Plot
    fig, axs = plt.subplots(figsize=(12, 13), ncols=2, nrows=5, dpi=300)
    axs = axs.ravel()
    
    plotted_data = []

    # Subplots
    for i in range(0, len(list_etype)):
    
        v = list_etype[i]
        if ctype == 'cveg':
            if v.startswith('glob'):
                vmax =  40  
            else:
                vmax = 200
        if ctype == 'csoil':
            vmax = 400
            
        coastline110.plot(ax=axs[i], color='#000000', linewidth=0.5) 
        
        ds_in.land_sea_mask.plot.imshow(
            ax=axs[i], 
            cmap='Greys', vmax=2, add_colorbar=False, alpha=0.5)
        
        im = ds_in[f'{ctype}_{v}'].plot.imshow(
            ax = axs[i], 
            vmin=0, vmax=vmax, cmap=cmap, add_colorbar=False
        )
    
        plotted_data.append(im)
        
        axs[i].set_title(ds_in[f'{ctype}_{v}'].attrs['ecosystem_type'], 
                         size='x-large')
        axs[i].axis('off')
        axs[i].set_ylim(-60, 85)
        
    
    # Colorbars
    if ctype == 'cveg':
        cbar_ax1 = fig.add_axes([0.92, 0.47, 0.01, 0.43])
        cbar1 = fig.colorbar(plotted_data[0], 
                             cax=cbar_ax1, 
                             label=r'Vegetation carbon (tC ha$^{-1}$)', 
                             extend='max')
    
        cbar_ax2 = fig.add_axes([0.92, 0.07, 0.01, 0.25])
        cbar2 = fig.colorbar(plotted_data[6], 
                             cax=cbar_ax2, 
                             label=r'Vegetation carbon (tC ha$^{-1}$)', 
                             extend='max')
    
    if ctype == 'csoil':
        cbar_ax1 = fig.add_axes([0.92, 0.28, 0.01, 0.43])
        cbar1 = fig.colorbar(plotted_data[0], 
                             cax=cbar_ax1, 
                             label=r'Soil organic carbon (tC ha$^{-1}$)', 
                             extend='max')
    
    # Layout
    fig.subplots_adjust(
        left=0.02,
        right=0.90,
        bottom=0.02,
        top=0.98,
        wspace=0.03,
        hspace=0.12,
    )
        
    # Export
    if ctype == 'cveg':
        fstr_out = 'fig_m01_maps_cveg'       
    if ctype == 'csoil':
        fstr_out = 'fig_m03_maps_csoil'
    
    fig.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300)
    fig.savefig(f'{dir08}png/{fstr_out}.png', dpi=300)
    

plot_maps('cveg', 'Greens')
plot_maps('csoil', 'Oranges')